# CrossMed 03 - Evaluate saved models
Load saved checkpoints one at a time and evaluate each on BUS-BRA, BrEaST, and BUSI.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from crossmed.experiments import load_experiment_matrix, run_experiment_matrix, aggregate_results
PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
MATRIX = PROJECT_ROOT / 'configs/experiment_matrix.yaml'
experiments = load_experiment_matrix(MATRIX)

## Select trained checkpoints

In [ ]:
SELECTED = ['resnet18_finetuned']
# SELECTED = list(experiments)
CLEANUP_BETWEEN_MODELS = True

## Evaluate
Existing `evaluation.json` files are skipped. To recompute evaluation, change `skip_completed` to `False`.

In [ ]:
comparison = run_experiment_matrix(
    MATRIX,
    selected=SELECTED,
    stage='evaluate',
    skip_completed=True,
    cleanup_between_runs=CLEANUP_BETWEEN_MODELS,
)
comparison

## Reload all available results without loading a model

In [ ]:
comparison = aggregate_results(MATRIX)
comparison

## Cross-dataset ROC-AUC comparison

In [ ]:
if not comparison.empty:
    auc_table = comparison.pivot(index='experiment', columns='dataset', values='roc_auc')
    display(auc_table)
    ax = auc_table.plot(kind='bar', figsize=(11, 5), ylim=(0, 1), rot=20)
    ax.set_ylabel('ROC-AUC')
    ax.set_title('CrossMed cross-dataset performance')
    plt.tight_layout()

## Other metrics

In [ ]:
if not comparison.empty:
    display(comparison[['experiment', 'dataset', 'balanced_accuracy', 'recall_sensitivity', 'specificity', 'f1', 'roc_auc']])